In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# Creates 2 qubits (q0=input, q1=ancilla), both starting at |0⟩. 1 classical bit, to store the one measurement you care about.
qc = QuantumCircuit(2, 1)

qc.x(1) # Flips q1 from |0⟩ to |1⟩
qc.h(0) # Turns input qubit into superposition (|0⟩+|1⟩)/√2
qc.h(1) # Turns the ancilla into (|0⟩−|1⟩)/√2

# The oracle for f(x)=x. Control=q0, target=q1: flips q1 whenever q0=1
qc.cx(0,1)

qc.h(0)

qc.measure(0,0)

print(qc.draw())

simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()
counts = result.get_counts()
print(counts)


     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├───────■──┤ H ├┤M├
     ├───┤┌───┐┌─┴─┐└───┘└╥┘
q_1: ┤ X ├┤ H ├┤ X ├──────╫─
     └───┘└───┘└───┘      ║ 
c: 1/═════════════════════╩═
                          0 
{'1': 1000}


In [7]:
def deutsch_circuit(oracle_type):
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    qc.h(0)
    qc.h(1)
    
    if oracle_type == 'f0':
        pass
    elif oracle_type == 'f1':
        qc.x(1)
    elif oracle_type == 'fx':
        qc.cx(0, 1)
    elif oracle_type == 'f1x':
        qc.x(0)
        qc.cx(0, 1)
        qc.x(0)
    
    qc.h(0)
    qc.measure(0, 0)
    return qc

simulator = AerSimulator()

for oracle in ['f0', 'f1', 'fx', 'f1x']:
    qc = deutsch_circuit(oracle)
    counts = simulator.run(qc, shots=1000).result().get_counts()
    print(f"{oracle}: {counts}")

f0: {'0': 1000}
f1: {'0': 1000}
fx: {'1': 1000}
f1x: {'1': 1000}


## Results
- f(x)=0 (constant) → measured 0 with certainty
- f(x)=1 (constant) → measured 0 with certainty  
- f(x)=x (balanced) → measured 1 with certainty
- f(x)=1-x (balanced) → measured 1 with certainty

Confirms Deutsch's algorithm: one oracle query distinguishes constant from balanced functions deterministically.